<a href="https://colab.research.google.com/github/muhammetalicvs-prog/flyrank-ml/blob/main/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage / Privacy Check

This notebook builds a temporally safer feature vector, documents each feature, attacks the feature set for leakage, and records all excluded fields.


## 1. Build the feature vector

I define the prediction moment as the beginning of the most recent 30-day window. The label represents whether the content was classified as declining during that following 30-day period.

To keep the feature vector temporally safer, I use activity from the previous 30-day window together with public-safe content and keyword metadata. I exclude the latest 30-day activity, direct label-source fields, identifiers, provider/model fields, and update-recency fields that may contain information from the outcome period.

Numeric missing values are accompanied by missingness indicators and filled with the median. Missing categorical values are represented as `unknown` and converted with one-hot encoding. Client and content identifiers are retained separately for grouping and traceability, but they are not included as model features.


In [1]:
import numpy as np
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "muhammetalicvs-prog/flyrank-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

df["is_declining_label"] = (
    df["trend_direction"].eq("down")
).astype("int8")

y = df["is_declining_label"].copy()

prev_impressions = (
    pd.to_numeric(df["impressions_prev_30d"], errors="coerce")
    .fillna(0)
    .clip(lower=0)
)

prev_clicks = (
    pd.to_numeric(df["clicks_prev_30d"], errors="coerce")
    .fillna(0)
    .clip(lower=0)
)

prev_sessions = (
    pd.to_numeric(df["sessions_prev_30d"], errors="coerce")
    .fillna(0)
    .clip(lower=0)
)

X_numeric = pd.DataFrame(index=df.index)

X_numeric["log_prev_impressions"] = np.log1p(prev_impressions)
X_numeric["log_prev_clicks"] = np.log1p(prev_clicks)
X_numeric["log_prev_sessions"] = np.log1p(prev_sessions)

X_numeric["prev_ctr_pct"] = (
    prev_clicks
    .div(prev_impressions.replace(0, np.nan))
    .mul(100)
    .fillna(0)
)

X_numeric["has_prev_impressions"] = (
    prev_impressions > 0
).astype("int8")

numeric_metadata = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]

for column in numeric_metadata:
    values = pd.to_numeric(df[column], errors="coerce")
    X_numeric[f"{column}_missing"] = values.isna().astype("int8")
    X_numeric[column] = values.fillna(values.median())

content_age = pd.to_numeric(
    df["content_age_days"],
    errors="coerce",
)

X_numeric["content_age_missing"] = (
    content_age.isna()
).astype("int8")

X_numeric["age_at_prediction_days"] = (
    content_age
    .sub(30)
    .clip(lower=0)
    .fillna(content_age.median())
)

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
]

categorical_frame = (
    df[categorical_features]
    .astype("string")
    .fillna("unknown")
)

X_categorical = pd.get_dummies(
    categorical_frame,
    prefix=categorical_features,
    dtype="int8",
)

X_safe = pd.concat(
    [X_numeric, X_categorical],
    axis=1,
)

feature_vector = pd.concat(
    [
        df[["content_id", "client_id"]],
        y.rename("is_declining_label"),
        X_safe,
    ],
    axis=1,
)

forbidden_features = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "provider_used",
    "model_used",
    "days_since_last_update",
}

forbidden_found = sorted(
    forbidden_features.intersection(X_safe.columns)
)

assert X_safe.shape[0] == df.shape[0]
assert X_safe.isna().sum().sum() == 0
assert forbidden_found == []

print("Source dataset shape:", df.shape)
print("Safe feature matrix shape:", X_safe.shape)
print("Feature-vector shape:", feature_vector.shape)
print("Positive-label base rate:", f"{y.mean():.2%}")
print("Remaining missing values:", int(X_safe.isna().sum().sum()))
print("Forbidden fields found in model features:", forbidden_found)

display(feature_vector.head())


Source dataset shape: (30000, 45)
Safe feature matrix shape: (30000, 29)
Feature-vector shape: (30000, 32)
Positive-label base rate: 54.21%
Remaining missing values: 0
Forbidden fields found in model features: []


,content_id,client_id,is_declining_label,log_prev_impressions,log_prev_clicks,log_prev_sessions,prev_ctr_pct,has_prev_impressions,search_volume_missing,search_volume,...,competition_level_MEDIUM,competition_level_unknown,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,content_304f48230142,client_f369cb89fc,1,6.895683,2.639057,2.302585,1.317123,1,0,10.0,...,0,0,0,0,1,0,0,0,1,0
1,content_a1fb4e703a9e,client_4e07408562,1,8.685416,0.693147,1.098612,0.016906,1,0,90.0,...,0,0,0,0,1,0,1,0,0,0
2,content_9aa793d4d895,client_7f2253d7e2,1,8.714403,1.386294,1.386294,0.049269,1,0,0.0,...,0,0,0,0,1,0,1,0,0,0
3,content_331d6c4de07b,client_19581e27de,0,8.344505,2.890372,3.295837,0.404184,1,0,10.0,...,0,0,0,0,1,1,0,0,0,0
4,content_d99b7a2d90ca,client_3fdba35f04,1,8.772300,1.098612,2.302585,0.030998,1,0,0.0,...,0,0,0,0,1,0,1,0,0,0


## 2. Feature notes

The safe feature matrix contains 29 model features. Previous-window impressions, clicks, sessions, and CTR describe activity that occurred before the prediction window. Continuous traffic variables are log-transformed to reduce the influence of extreme values.

Keyword and content metadata are assumed to be available at the prediction moment. Numeric metadata is converted to numeric form, accompanied by a missing-value indicator, and filled with the median. Categorical values are filled with `unknown` and converted into one-hot indicator columns.

`age_at_prediction_days` estimates content age at the beginning of the outcome window by subtracting 30 days from the supplied content-age value. Identifiers are retained outside the feature matrix, while current-window activity, direct label sources, provider fields, and potentially post-prediction freshness fields are excluded.


In [2]:
feature_notes_rows = []

for feature in X_safe.columns:

    if feature == "log_prev_impressions":
        feature_type = "numeric"
        meaning = "Natural-log transformed impressions from the previous 30-day observation window."
        missing_handling = "Invalid or missing source values are treated as zero before log1p transformation."
        available_when = "Available before the outcome window begins."

    elif feature == "log_prev_clicks":
        feature_type = "numeric"
        meaning = "Natural-log transformed clicks from the previous 30-day observation window."
        missing_handling = "Invalid or missing source values are treated as zero before log1p transformation."
        available_when = "Available before the outcome window begins."

    elif feature == "log_prev_sessions":
        feature_type = "numeric"
        meaning = "Natural-log transformed sessions from the previous 30-day observation window."
        missing_handling = "Invalid or missing source values are treated as zero before log1p transformation."
        available_when = "Available before the outcome window begins."

    elif feature == "prev_ctr_pct":
        feature_type = "numeric"
        meaning = "Previous-window click-through rate calculated as clicks divided by impressions, expressed as a percentage."
        missing_handling = "Set to zero when previous impressions are zero or unavailable."
        available_when = "Available before the outcome window begins."

    elif feature == "has_prev_impressions":
        feature_type = "binary"
        meaning = "Indicates whether the content received at least one impression in the previous observation window."
        missing_handling = "Missing impressions are treated as zero."
        available_when = "Available before the outcome window begins."

    elif feature == "age_at_prediction_days":
        feature_type = "numeric"
        meaning = "Estimated content age at the start of the outcome window, calculated as content_age_days minus 30."
        missing_handling = "Missing values are filled with the median content age."
        available_when = "Derived to represent information available at prediction time."

    elif feature == "content_age_missing":
        feature_type = "binary"
        meaning = "Indicates that the original content-age value was missing."
        missing_handling = "The feature itself records missingness."
        available_when = "Available at prediction time."

    elif feature.endswith("_missing"):
        original_feature = feature.removesuffix("_missing")
        feature_type = "binary"
        meaning = f"Indicates that the original {original_feature} value was missing."
        missing_handling = "The feature itself records missingness."
        available_when = "Based on metadata assumed to be available at prediction time."

    elif feature in {
        "search_volume",
        "competition",
        "cpc",
        "word_count",
        "char_count",
    }:
        feature_type = "numeric"
        meaning_map = {
            "search_volume": "Estimated search demand for the target keyword.",
            "competition": "Numeric keyword competition measure.",
            "cpc": "Estimated keyword cost-per-click value.",
            "word_count": "Number of words in the content record.",
            "char_count": "Number of characters in the content record.",
        }
        meaning = meaning_map[feature]
        missing_handling = "Converted to numeric form and filled with the column median."
        available_when = "Metadata assumed to be available before prediction."

    elif feature.startswith("competition_level_"):
        category = feature.replace("competition_level_", "")
        feature_type = "one-hot categorical"
        meaning = f"Indicates that keyword competition level is '{category}'."
        missing_handling = "Missing categories are represented by the 'unknown' indicator."
        available_when = "Keyword metadata assumed to be available before prediction."

    elif feature.startswith("content_type_"):
        category = feature.replace("content_type_", "")
        feature_type = "one-hot categorical"
        meaning = f"Indicates that the content type is '{category}'."
        missing_handling = "Missing categories are represented by the 'unknown' indicator."
        available_when = "Content metadata assumed to be available before prediction."

    elif feature.startswith("main_intent_"):
        category = feature.replace("main_intent_", "")
        feature_type = "one-hot categorical"
        meaning = f"Indicates that the main search intent is '{category}'."
        missing_handling = "Missing categories are represented by the 'unknown' indicator."
        available_when = "Intent metadata assumed to be available before prediction."

    else:
        feature_type = "review required"
        meaning = "Feature meaning was not automatically assigned."
        missing_handling = "Review required."
        available_when = "Review required."

    feature_notes_rows.append(
        {
            "feature": feature,
            "type": feature_type,
            "meaning": meaning,
            "missing_handling": missing_handling,
            "available_when": available_when,
        }
    )

feature_notes = pd.DataFrame(feature_notes_rows)

documented_features = set(feature_notes["feature"])
model_features = set(X_safe.columns)

undocumented_features = sorted(model_features - documented_features)
unexpected_notes = sorted(documented_features - model_features)

assert undocumented_features == []
assert unexpected_notes == []
assert len(feature_notes) == X_safe.shape[1]

print("Model features:", X_safe.shape[1])
print("Documented features:", len(feature_notes))
print("Undocumented features:", undocumented_features)
print("Unexpected documentation rows:", unexpected_notes)

display(feature_notes)


Model features: 29
Documented features: 29
Undocumented features: []
Unexpected documentation rows: []


,feature,type,meaning,missing_handling,available_when
0,log_prev_impressions,numeric,Natural-log transformed impressions from the p...,Invalid or missing source values are treated a...,Available before the outcome window begins.
1,log_prev_clicks,numeric,Natural-log transformed clicks from the previo...,Invalid or missing source values are treated a...,Available before the outcome window begins.
2,log_prev_sessions,numeric,Natural-log transformed sessions from the prev...,Invalid or missing source values are treated a...,Available before the outcome window begins.
3,prev_ctr_pct,numeric,Previous-window click-through rate calculated ...,Set to zero when previous impressions are zero...,Available before the outcome window begins.
4,has_prev_impressions,binary,Indicates whether the content received at leas...,Missing impressions are treated as zero.,Available before the outcome window begins.
5,search_volume_missing,binary,Indicates that the original search_volume valu...,The feature itself records missingness.,Based on metadata assumed to be available at p...
6,search_volume,numeric,Estimated search demand for the target keyword.,Converted to numeric form and filled with the ...,Metadata assumed to be available before predic...
7,competition_missing,binary,Indicates that the original competition value ...,The feature itself records missingness.,Based on metadata assumed to be available at p...
8,competition,numeric,Numeric keyword competition measure.,Converted to numeric form and filled with the ...,Metadata assumed to be available before predic...
9,cpc_missing,binary,Indicates that the original cpc value was miss...,The feature itself records missingness.,Based on metadata assumed to be available at p...


## 3. The leakage hunt

I attacked the feature set from three directions: direct label sources, future or outcome-window measurements, and operational product fields. `trend_direction` deterministically defines the binary label, so using it or `trend_pct` would reveal the answer rather than predict it. Fields from the latest 30-day window were excluded because they belong to the period being predicted, while provider and model fields were excluded because they describe the data-generation product rather than the content opportunity.

The checks below search both the original columns and the engineered feature names. The measured result should show a 100% match between `trend_direction` and the label, while showing that none of the identified leakage candidates entered the final feature matrix.


In [3]:
direct_label_candidates = [
    column
    for column in [
        "is_declining_label",
        "trend_direction",
        "trend_pct",
    ]
    if column in df.columns
]

future_window_candidates = sorted(
    column
    for column in df.columns
    if column.endswith("_last_30d")
)

product_field_candidates = [
    column
    for column in [
        "provider_used",
        "model_used",
    ]
    if column in df.columns
]

identifier_candidates = [
    column
    for column in [
        "content_id",
        "client_id",
    ]
    if column in df.columns
]

freshness_candidates = [
    column
    for column in [
        "days_since_last_update",
    ]
    if column in df.columns
]

leakage_groups = {
    "direct label source": direct_label_candidates,
    "future/outcome window": future_window_candidates,
    "product or generation field": product_field_candidates,
    "identifier": identifier_candidates,
    "timing-sensitive freshness field": freshness_candidates,
}

leakage_rows = []

for category, columns in leakage_groups.items():
    for column in columns:
        leakage_rows.append(
            {
                "field": column,
                "risk_category": category,
                "present_in_source": column in df.columns,
                "present_in_model_features": column in X_safe.columns,
            }
        )

leakage_audit = pd.DataFrame(leakage_rows)

label_recreated_from_trend = (
    df["trend_direction"]
    .eq("down")
    .astype("int8")
)

label_source_match_rate = (
    label_recreated_from_trend
    .eq(y)
    .mean()
)

all_leakage_candidates = sorted(
    {
        column
        for columns in leakage_groups.values()
        for column in columns
    }
)

leakage_fields_in_model = sorted(
    set(all_leakage_candidates)
    .intersection(X_safe.columns)
)

risky_name_tokens = [
    "trend",
    "last_30d",
    "provider_used",
    "model_used",
    "days_since_last_update",
    "is_declining",
]

suspicious_engineered_features = sorted(
    feature
    for feature in X_safe.columns
    if any(
        token in feature.lower()
        for token in risky_name_tokens
    )
)

assert label_source_match_rate == 1.0
assert leakage_fields_in_model == []
assert suspicious_engineered_features == []

print("Label match from trend_direction:", f"{label_source_match_rate:.2%}")
print("Exact leakage fields found in model:", leakage_fields_in_model)
print("Suspicious engineered feature names:", suspicious_engineered_features)

display(leakage_audit)


Label match from trend_direction: 100.00%
Exact leakage fields found in model: []
Suspicious engineered feature names: []


,field,risk_category,present_in_source,present_in_model_features
0,is_declining_label,direct label source,True,False
1,trend_direction,direct label source,True,False
2,trend_pct,direct label source,True,False
3,clicks_last_30d,future/outcome window,True,False
4,impressions_last_30d,future/outcome window,True,False
5,sessions_last_30d,future/outcome window,True,False
6,provider_used,product or generation field,True,False
7,model_used,product or generation field,True,False
8,content_id,identifier,True,False
9,client_id,identifier,True,False


## 4. What I excluded and why

I excluded every field whose timing, privacy status, or relationship to the target made it unsafe for this prediction task. Client and content identifiers remain available only for grouped validation and traceability. Direct label sources, latest-window measurements, operational provider fields, and timing-sensitive freshness values are not model inputs.

I also excluded aggregated performance fields when their exact observation window could not be placed safely before the prediction moment. This is intentionally conservative: removing a potentially useful field is preferable to reporting an inflated result caused by leakage. The resulting recommendations should therefore be interpreted as offline, directional decision support rather than proof of causal content impact.


In [4]:
used_raw_sources = {
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "competition_level",
    "content_type",
    "main_intent",
}

def classify_exclusion(column):
    lower_column = column.lower()

    if column in {"content_id", "client_id"}:
        return (
            "identifier/grouping field",
            "Retained only for traceability and client-grouped validation; not used as a predictive feature.",
        )

    if column == "is_declining_label":
        return (
            "target",
            "This is the outcome being predicted and cannot be used as an input.",
        )

    if column in {"trend_direction", "trend_pct"}:
        return (
            "direct label source",
            "This field directly defines or quantifies the decline label.",
        )

    if column.endswith("_last_30d"):
        return (
            "future/outcome-window measurement",
            "This value belongs to the latest 30-day period represented by the prediction target.",
        )

    if column in {"provider_used", "model_used"}:
        return (
            "product/generation field",
            "This describes how the record was generated rather than a stable content opportunity signal.",
        )

    if column == "days_since_last_update":
        return (
            "timing-sensitive freshness field",
            "Its value may have been calculated after the prediction moment, so temporal safety was not established.",
        )

    if (
        column.endswith("_90d")
        or column
        in {
            "ctr",
            "avg_position",
            "engagement_rate",
        }
    ):
        return (
            "temporally ambiguous aggregate",
            "The aggregation window may overlap the outcome period, so it was excluded conservatively.",
        )

    if any(
        token in lower_column
        for token in [
            "client_name",
            "domain",
            "url",
            "private_query",
            "raw_query",
        ]
    ):
        return (
            "privacy or identification risk",
            "The field may reveal a client, domain, URL, or private query.",
        )

    return (
        "not selected",
        "Availability before the prediction moment or stable decision value was not sufficiently established for this audit.",
    )

excluded_rows = []

for column in df.columns:
    if column in used_raw_sources:
        continue

    category, reason = classify_exclusion(column)

    excluded_rows.append(
        {
            "field": column,
            "category": category,
            "decision": "exclude from model features",
            "reason": reason,
        }
    )

exclusion_register = (
    pd.DataFrame(excluded_rows)
    .sort_values(
        ["category", "field"],
        ignore_index=True,
    )
)

excluded_field_names = set(exclusion_register["field"])

required_exclusions = {
    column
    for column in [
        "content_id",
        "client_id",
        "is_declining_label",
        "trend_direction",
        "trend_pct",
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "provider_used",
        "model_used",
        "days_since_last_update",
    ]
    if column in df.columns
}

assert used_raw_sources.issubset(df.columns)
assert required_exclusions.issubset(excluded_field_names)
assert set(X_safe.columns).isdisjoint(required_exclusions)

print("Raw fields used to build features:", len(used_raw_sources))
print("Source fields excluded:", len(exclusion_register))
print("Required unsafe fields excluded:", required_exclusions.issubset(excluded_field_names))
print("Unsafe fields present in final matrix:", sorted(required_exclusions.intersection(X_safe.columns)))

display(exclusion_register)


Raw fields used to build features: 12
Source fields excluded: 33
Required unsafe fields excluded: True
Unsafe fields present in final matrix: []


,field,category,decision,reason
0,trend_direction,direct label source,exclude from model features,This field directly defines or quantifies the ...
1,trend_pct,direct label source,exclude from model features,This field directly defines or quantifies the ...
2,clicks_last_30d,future/outcome-window measurement,exclude from model features,This value belongs to the latest 30-day period...
3,impressions_last_30d,future/outcome-window measurement,exclude from model features,This value belongs to the latest 30-day period...
4,sessions_last_30d,future/outcome-window measurement,exclude from model features,This value belongs to the latest 30-day period...
5,client_id,identifier/grouping field,exclude from model features,Retained only for traceability and client-grou...
6,content_id,identifier/grouping field,exclude from model features,Retained only for traceability and client-grou...
7,age_tier,not selected,exclude from model features,Availability before the prediction moment or s...
8,age_tier_order,not selected,exclude from model features,Availability before the prediction moment or s...
9,ai_traffic_pct,not selected,exclude from model features,Availability before the prediction moment or s...


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.